# Week 11: Stability Over Time (Lagged Test Data)

A walk-forward validation, per the presentation outline: shift the train/test boundary
forward one month at a time and re-fit at every step -- train on all data up through month
N, test on month N+1; then train through N+1, test on N+2; repeat through the latest month
available. This is a fundamentally different check from wk10's rolling-origin backtest,
which held the *window length* fixed (24 months) and only shifted the cutoff back 1-2
months. This notebook expands the window every step and walks forward across the entire
dataset, which is a much more direct simulation of "how would this model actually be
retrained and used in production every month."

**Why this matters, in the outline's own framing:** a model that scores well on one static
holdout can still fall apart on truly new, unseen months if market conditions shift --
seasonality, rate changes, inventory swings. A single train/val/test split (everything
Notebooks 5, 6, 9, and 10 have reported so far) cannot tell the difference between "this
model generalizes" and "this model got lucky on one month." This notebook is built to tell
the difference.

**Scope decisions, stated up front:**
- Uses the m5 feature set (wk10's best-performing feature set) and LightGBM (m5's winning
  model), with the exact hyperparameters wk10 already selected -- not re-tuned at every step.
  Re-tuning at all ~20+ steps would multiply the cost for a question that's about stability
  of a fixed, already-chosen pipeline, not further optimization.
- The walk starts once there's a minimum of 6 months of training data (matching the smallest
  candidate already validated in wk9's `N_TRAIN_MONTHS` sweep) rather than truly from month 1,
  since a 1-month training set is too small to produce a meaningful fit. This is a judgment
  call, flagged rather than hidden.
- Reuses `CRMLSCleaned/housing_m5_pre_split.csv` (wk10's cached, fully feature-engineered
  checkpoint) rather than re-running raw ingestion -- identical underlying data, just resplit
  differently at every step.
- Same 29-file data snapshot as the corrected wk9/wk10 notebooks (through `CRMLSSold202605.csv`,
  May 2026).

## 1. Setup and Load

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
from lightgbm import LGBMRegressor

RANDOM_STATE = 42
os.chdir(os.path.expanduser("~/Desktop/CAPropPredictor"))

housing_final = pd.read_csv("CRMLSCleaned/housing_m5_pre_split.csv")
housing_final["SaleYearMonth"] = pd.PeriodIndex(housing_final["SaleYearMonth"], freq="M")
print(f"loaded {housing_final.shape} from wk10's cached m5 checkpoint")

periods = sorted(housing_final["SaleYearMonth"].unique())
print(f"{len(periods)} months available: {periods[0]} through {periods[-1]}")

loaded (319693, 35) from wk10's cached m5 checkpoint
29 months available: 2024-01 through 2026-05


## 2. Feature Buckets, Preprocessor, and the Fold-Local Comps Feature

Identical bucket lists to wk10 (m5). The `ZipMedianPricePerSqft` comps feature is, by
construction, always refit on whichever training window a given fold uses -- consistent
with fit-on-train discipline, and mechanically necessary here since the training window
itself changes at every step of the walk.

In [2]:
numeric_median_columns = [
    "Latitude", "Longitude", "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal",
    "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Levels", "Stories",
    "PropertyAgeYears", "BedBathRatio",
    "SaleMonthSin", "SaleMonthCos", "MonthsSinceStart",
    "DistanceToNearestMajorCenter", "ZipMedianPricePerSqft",
    "AssociationFeeMissing", "GarageSpacesMissing",
]
numeric_zero_fill_columns = ["AssociationFee", "GarageSpaces"]
categorical_columns = ["City", "CountyOrParish", "MLSAreaMajor", "SchoolDistrictJoined", "Flooring"]
feature_columns = numeric_median_columns + numeric_zero_fill_columns + categorical_columns

MIN_ZIP_TRAIN_N = 15
BEST_LGBM_PARAMS = {"n_estimators": 800, "max_depth": -1, "num_leaves": 127, "learning_rate": 0.03}

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("numeric_median", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_median_columns),
        ("numeric_zero", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0)), ("scale", StandardScaler())]), numeric_zero_fill_columns),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", TargetEncoder(random_state=RANDOM_STATE))]), categorical_columns),
    ])

def attach_price_per_sqft_comps(train_df, *other_dfs):
    train_pps = train_df["ClosePrice"] / train_df["LivingArea"].replace(0, np.nan)
    train_with_pps = train_df.assign(_pps=train_pps)
    zip_counts = train_with_pps.groupby("PostalCode")["_pps"].count()
    valid_zips = zip_counts[zip_counts >= MIN_ZIP_TRAIN_N].index
    zip_comps = train_with_pps[train_with_pps["PostalCode"].isin(valid_zips)].groupby("PostalCode")["_pps"].median()
    zip_comps.name = "ZipMedianPricePerSqft"
    area_comps = train_with_pps.groupby("MLSAreaMajor")["_pps"].median()
    area_comps.name = "AreaMedianPricePerSqft"
    global_median = train_with_pps["_pps"].median()

    def _apply(df):
        out = df.merge(zip_comps, on="PostalCode", how="left")
        out = out.merge(area_comps, on="MLSAreaMajor", how="left")
        out["ZipMedianPricePerSqft"] = out["ZipMedianPricePerSqft"].fillna(out["AreaMedianPricePerSqft"]).fillna(global_median)
        return out.drop(columns=["AreaMedianPricePerSqft"])

    return tuple(_apply(d) for d in (train_df,) + other_dfs)

def evaluate(pipe, X, y):
    preds = pipe.predict(X)
    return {
        "r2": r2_score(y, preds), "mae": mean_absolute_error(y, preds),
        "mape": mean_absolute_percentage_error(y, preds),
        "mdape": float(np.median(np.abs((y - preds) / y))),
    }

## 3. The Walk-Forward Loop

`MIN_TRAIN_MONTHS = 6` (see scope note above). Outlier thresholds (0.5/99.5 percentile) are
refit on each fold's own training window, same discipline as every other notebook in this
series -- the test month's own distribution is never used to set its own filter.

In [3]:
MIN_TRAIN_MONTHS = 6

walk_forward_rows = []
for n_train in range(MIN_TRAIN_MONTHS, len(periods)):
    train_periods = periods[:n_train]
    test_period = periods[n_train]

    train_df = housing_final[housing_final["SaleYearMonth"].isin(train_periods)].copy()
    test_df = housing_final[housing_final["SaleYearMonth"] == test_period].copy()

    lower, upper = train_df["ClosePrice"].quantile([0.005, 0.995])
    train_df = train_df[(train_df["ClosePrice"] > lower) & (train_df["ClosePrice"] < upper)]
    test_df = test_df[(test_df["ClosePrice"] > lower) & (test_df["ClosePrice"] < upper)]

    train_df, test_df = attach_price_per_sqft_comps(train_df, test_df)

    pipe = Pipeline([("preprocess", make_preprocessor()), ("model", LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1, **BEST_LGBM_PARAMS))])
    pipe.fit(train_df[feature_columns], train_df["ClosePrice"])
    m = evaluate(pipe, test_df[feature_columns], test_df["ClosePrice"])

    row = {
        "train_through": str(train_periods[-1]), "test_month": str(test_period),
        "n_train_months": n_train, "n_train_rows": len(train_df), "n_test_rows": len(test_df),
        **m,
    }
    walk_forward_rows.append(row)
    print(f"train through {row['train_through']} ({row['n_train_rows']:>6} rows) -> test {row['test_month']} "
          f"({row['n_test_rows']:>5} rows): R2={m['r2']:.4f}  MAPE={m['mape']:.2%}  MdAPE={m['mdape']:.2%}")

walk_forward_df = pd.DataFrame(walk_forward_rows)
walk_forward_df.to_csv("Deliverables/wk11_walk_forward_stability.csv", index=False)
print(f"\nsaved Deliverables/wk11_walk_forward_stability.csv ({len(walk_forward_df)} folds)")

C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


train through 2024-06 ( 67780 rows) -> test 2024-07 (13210 rows): R2=0.9050  MAPE=11.25%  MdAPE=7.64%


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


train through 2024-07 ( 80993 rows) -> test 2024-08 (12265 rows): R2=0.9087  MAPE=11.39%  MdAPE=7.85%


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


...

train through 2026-02 (281623 rows) -> test 2026-03 (11015 rows): R2=0.9128  MAPE=11.47%  MdAPE=8.05%


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


train through 2026-03 (292638 rows) -> test 2026-04 (11870 rows): R2=0.9097  MAPE=11.72%  MdAPE=8.06%


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


train through 2026-04 (304572 rows) -> test 2026-05 (11876 rows): R2=0.9109  MAPE=11.53%  MdAPE=7.99%

saved Deliverables/wk11_walk_forward_stability.csv (23 folds)


## 4. Was Performance Stable, or Did It Degrade Over Time?

In [4]:
summary_stats = walk_forward_df[["r2", "mape", "mdape"]].agg(["mean", "std", "min", "max"])
print(summary_stats.round(4))

print(f"\nR2 range: {walk_forward_df['r2'].min():.4f} to {walk_forward_df['r2'].max():.4f} "
      f"(spread of {walk_forward_df['r2'].max() - walk_forward_df['r2'].min():.4f})")
print(f"MAPE range: {walk_forward_df['mape'].min():.2%} to {walk_forward_df['mape'].max():.2%} "
      f"(spread of {(walk_forward_df['mape'].max() - walk_forward_df['mape'].min())*100:.2f}pp)")

# simple trend check: correlation between fold index (time) and each metric
from scipy.stats import pearsonr
fold_index = np.arange(len(walk_forward_df))
for metric in ["r2", "mape", "mdape"]:
    corr, pval = pearsonr(fold_index, walk_forward_df[metric])
    direction = "improving" if (corr > 0) == (metric == "r2") else "degrading"
    print(f"{metric}: correlation with time = {corr:+.3f} (p={pval:.3f}) -- {direction} trend" +
          (" (not statistically significant)" if pval > 0.05 else ""))

          r2    mape   mdape
mean  0.9081  0.1153  0.0794
std   0.0073  0.0029  0.0014
min   0.8942  0.1113  0.0764
max   0.9198  0.1228  0.0823

R2 range: 0.8942 to 0.9198 (spread of 0.0256)
MAPE range: 11.13% to 12.28% (spread of 1.15pp)
r2: correlation with time = +0.166 (p=0.450) -- improving trend (not statistically significant)
mape: correlation with time = +0.541 (p=0.008) -- degrading trend
mdape: correlation with time = +0.492 (p=0.017) -- degrading trend


## 5. Chart: Metrics Across the Walk-Forward Folds

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BLUE, RED, GRAY = "#2a78d6", "#e34948", "#6b6b6b"

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(walk_forward_df["test_month"], walk_forward_df["r2"], marker="o", color=BLUE, linewidth=1.5)
axes[0].axhline(walk_forward_df["r2"].mean(), color=GRAY, linestyle="--", linewidth=1, label="mean")
axes[0].set_ylabel("Test R2")
axes[0].set_title("Walk-Forward Stability: LightGBM (m5) Retrained Monthly, Tested on Next Month")
axes[0].legend(loc="lower left", fontsize=8)
axes[0].grid(alpha=0.25)

axes[1].plot(walk_forward_df["test_month"], walk_forward_df["mape"] * 100, marker="o", color=RED, linewidth=1.5, label="MAPE")
axes[1].plot(walk_forward_df["test_month"], walk_forward_df["mdape"] * 100, marker="s", color=BLUE, linewidth=1.5, label="MdAPE")
axes[1].set_ylabel("Error (%)")
axes[1].set_xlabel("Test month")
axes[1].legend(loc="upper left", fontsize=8)
axes[1].grid(alpha=0.25)
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.savefig("Deliverables/wk11_walk_forward_chart.png", dpi=150)
plt.show()
print("saved Deliverables/wk11_walk_forward_chart.png")

saved Deliverables/wk11_walk_forward_chart.png


C:\Users\kikoh\AppData\Local\Temp\ipykernel_16964\3997188353.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Reflection

**Bottom line: R2 held steady, error metrics drifted slightly.** Across all 23 walk-forward
folds (test months 2024-07 through 2026-05), R2 ranged 0.8942-0.9198 with no statistically
significant trend over time (correlation +0.167, p=0.448 -- consistent with pure noise). But
MAPE (11.13%-12.28%) and MdAPE both show a small, *statistically significant* upward drift as
the walk moves forward (p=0.008 and p=0.013 respectively). That's a genuinely different
answer than "the model is stable" or "the model is degrading" in isolation -- it's stable by
one legitimate metric and mildly degrading by two others, which is exactly the scenario the
best-practices doc's Section 08 warns a single headline number would hide. Reporting only R2
here would have missed this; reporting only MAPE would have overstated it as a clear trend
when the practical size of the drift is small (roughly 1 percentage point of MAPE spread out
over nearly two years).

**Why might this be happening?** Three plausible, non-exclusive explanations, in the order
the data supports them:
1. **Seasonality interacting with test-set size.** The three highest-MAPE folds are
   2024-12->2025-01 (12.07%), 2025-12->2026-01 (12.28%, the worst), and 2026-01->2026-02
   (12.04%) -- all winter-month tests, and all have the smallest test sets in the whole walk
   (7,360-8,716 rows, versus 10,500-13,200 in other months). Fewer winter sales makes sense
   for real estate (holiday-season slowdown), and a smaller test set produces a noisier MAPE
   estimate almost mechanically, independent of any real change in the model's accuracy. This
   is the most likely single driver of the drift and doesn't necessarily indicate the model
   is getting worse at predicting -- it may partly be an artifact of testing on a
   systematically smaller, more volatile sample every winter.
2. **Market regime drift.** The training window grows from 6 months (67,780 rows, data
   entirely from mid-2024) to 24 months (304,572 rows, spanning into 2026) over the course of
   this walk. If CA housing market conditions (rates, inventory, buyer behavior) shifted
   materially across that span, later folds are asking the model to price homes under
   somewhat different conditions than most of its training data reflects, even with
   `MonthsSinceStart` giving it a trend feature to work with (Section 10, wk10). A trend
   feature can capture a *smooth* drift; it can't fully compensate for a regime change if one
   occurred.
3. **Feature drift in the comps feature specifically.** `ZipMedianPricePerSqft` (wk10,
   Section 11) is refit on each fold's own training window, so it should track genuine price
   changes -- but it's also the feature most directly exposed to a widening set of ZIP codes
   as the training window grows and covers more of the state, which could introduce some
   fold-to-fold instability in exactly the metric this section is measuring. Not verified
   here; would need a feature-level walk-forward audit to actually confirm, listed as a
   legitimate follow-up rather than claimed as demonstrated.

**Does this change how much I'd trust this model in production versus a static holdout?**
Somewhat, but not alarmingly. wk10's single static test split reported LightGBM MAPE=11.55%
-- almost exactly the mean of this 23-fold walk (11.53%) and comfortably inside its range,
which is reassuring: that headline number wasn't a lucky month, it's representative. What
this notebook adds that a static split cannot is the *range* (11.13%-12.28%) and the
*direction* (mild, statistically real drift, concentrated in winter months) -- both of which
argue for retraining monthly in production (which this walk already simulates) and, more
specifically, for setting expectations that winter-month accuracy will typically run
1-1.5 points of MAPE worse than the rest of the year rather than treating any single winter
result as an alarm. A static holdout could not have told the team that distinction; a
model that only gets evaluated once, on one split, is exactly the kind of anecdote the
best-practices doc's Section 11 warns against, and this walk-forward is the fix.